### Welcome and Setup

This notebook demonstrates the application of LIME to a BERT-based text classifier.<br>
The classifier predicts the whether or not a snippet of code is human-authored or machined-generated and visually explains the predictions.<br>
(See Orel et al. (2025) "𝙳𝚛𝚘𝚒𝚍 : A Resource Suite for AI-Generated Code Detection" https://doi.org/10.48550/arXiv.2507.10583).<br>

We demonstrate both, the use of the official lime package and a custom version in more detail.<br>
In the last section of this notebook you execute a user interface to control LIME settings, skip through the dataset, and see iterative updates of an explanation heatmap as the number of samples grows. 

This notebook was originally inspired by a LIME demo by [Cristian Arteaga](https://arteagac.github.io) available for [Colab](https://nbviewer.org/url/arteagac.github.io/blog/lime_image.ipynb). 


In [1]:
# Welcome!
# Please setup the following venv for this notebook from you terminal first:
r'''
python3 -m venv .venv3
# or on Windows:
# py -3 -m venv .venv3
'''

# activate it:
r'''
source .venv3/bin/activate
# Windows PowerShell:
# .venv3\Scripts\Activate.ps1
'''

# Then install the necessary dependancies:
r'''
pip install --upgrade pip

# If you have cuda support, install your appropriate torch version
# (check your NVIDIA driver with the nvidia-smi command to determine your compatible version)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# specific kernel and widgets version for compatibility
pip install "ipykernel<7" "ipywidgets>=7.6,<8"

# jupyter server alternatives
pip install notebook jupyterlab

# our libs
pip install lime scikit-learn datasets pygments
pip install --upgrade "transformers" "accelerate"
''';

# And dont forget to select this venv as the notebook kernel.
# We recommend using jupyter lab for the best experience.

In [2]:
print("Starting...")

### Imports
import html, random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import pairwise_distances
from sklearn.linear_model import LinearRegression, Ridge, Lasso

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pygments import highlight
from pygments.lexers import get_lexer_by_name
from pygments.formatters import HtmlFormatter
from pygments.token import Token, Comment, Literal

Starting...


## LIME for Code classification

In [3]:
### Config
DATASET_NAME  = "project-droid/DroidCollection"
DATASET_SPLIT = "train"
CODE_COL      = "Code"
LABEL_COL     = "Label"

# CHOSE MODEL VARIANT HERE
# You can see all Droid Classifiers on hf here: https://huggingface.co/collections/project-droid/droid
DROID_REPO    = "project-droid/DroidDetect-Base-Binary"  # "project-droid/DroidDetect-Base-Binary" or "project-droid/DroidDetect-Large-Binary"
BASE_MODEL    = "answerdotai/ModernBERT-base"  # "answerdotai/ModernBERT-base" or "answerdotai/ModernBERT-large"

PROJECTION_DIM         = 256
NUM_CLASSES            = 2
DEFAULT_MACHINE_SAMPLE = 11  # default sample index on machine-only subset for init
MAX_SEQ_LEN            = 512
BATCH_SIZE_DEFAULT     = 32

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu"))

print(f"Using device: {device}")

Using device: cuda


### Data and Classifier: DroidCollection, DroidDetect

In [4]:
### Dataset

# Load DroidCollection as binary dataset of human-vs-machine generated code
def _map_label(row):
    lab = row[LABEL_COL]
    return {"y": 0 if lab == "HUMAN_GENERATED" else 1}

print("Loading DroidCollection dataset...")
raw_ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

print("Mapping labels to binary classification targets (0=HUMAN, 1=MACHINE)...")
ds = raw_ds.map(_map_label, desc="Mapping labels")

# Retieve one default sample (machine generated)
machine_indices = [i for i, r in enumerate(ds) if r["y"] == 1]
sample_idx = machine_indices[DEFAULT_MACHINE_SAMPLE]
x_example = ds[sample_idx][CODE_COL]
y_example = ds[sample_idx]["y"]
LANG = ds[sample_idx].get("Language", "python")

# Info
print(f"Sample index: {sample_idx}")
print(f"Label (0=HUMAN, 1=MACHINE): {y_example}")
print("\nTruncaded code snippet:")
print("==="*15)
print(x_example[:200], "...\n")
print("==="*15)

### Model

# The DroidDetector backbone from ... is a finetuned ModernBERT encoder (see ...) with BPE-tokenizer. # TODO
print("Loading tokenizer and ModernBERT encoder...")
tokenizer    = AutoTokenizer.from_pretrained(DROID_REPO)
text_encoder = AutoModel.from_pretrained(BASE_MODEL)
TEXT_EMBEDDING_DIM = text_encoder.config.hidden_size  # eg. 768 for DroidDetect-Base-Binary

class DroidDetector(nn.Module):
    """ModernBERT encoder + projection + classifier for binary HUMAN/MACHINE."""
    def __init__(self, text_encoder, projection_dim: int = PROJECTION_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.text_encoder    = text_encoder
        self.text_projection = nn.Linear(TEXT_EMBEDDING_DIM, projection_dim)
        self.classifier      = nn.Linear(projection_dim, num_classes)

    def forward(self, input_ids=None, attention_mask=None):
        enc_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = enc_out.last_hidden_state.mean(dim=1)  # mean pool over T
        projected_text = F.relu(self.text_projection(sentence_embeddings))
        logits = self.classifier(projected_text)
        return logits

print("Loading DroidDetect checkpoint/weights...")
model = DroidDetector(text_encoder).to(device)
ckpt_path = hf_hub_download(DROID_REPO, "pytorch_model.bin")
state_dict = torch.load(ckpt_path, map_location=device)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys from checkpoint:", missing)
# print("Unexpected keys:", unexpected)  # we dont need the training-specific modules

model.eval()
labels = ["HUMAN_GENERATED", "MACHINE_GENERATED"]

### Predictions 
@torch.no_grad()
def detect_ai(snippets: list[str]) -> np.ndarray:
    """
    snippets: list of code strings to classify
    returns: probabilities of shape (N, num_classes)
    """
    # the hf tokenizer returns a tensors of shape (batch_size, seq_len)
    enc = tokenizer(
        snippets,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    logits = model(**enc)  # the model processes the snippets as one batch
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs

def detect_ai_batched(
    snippets: list[str],
    batch_size: int = BATCH_SIZE_DEFAULT,
    show_progress: bool = False,
    desc: str | None = None) -> np.ndarray:
    """
    Splits a list of snippets to apply `code_detect_predict` on batches with optional progress bar.
    """
    indices = range(0, len(snippets), batch_size)
    if show_progress:
        indices = tqdm(indices, desc=desc or "Batches", leave=False)

    batches = [detect_ai(snippets[i:i + batch_size]) for i in indices]
    return np.vstack(batches)


# Run example classification
print("Running classification on example...")
probs = detect_ai([x_example])[0]
top_pred_classes = probs.argsort()[::-1]  # 2 classes from DroidDetector

print("\nPredictions:")
for i in top_pred_classes:
    print(f"  {i} - {labels[i]}: {float(probs[i]):.4f}")
print(f"GT label: {y_example}")

Loading DroidCollection dataset...
Mapping labels to binary classification targets (0=HUMAN, 1=MACHINE)...
Sample index: 14
Label (0=HUMAN, 1=MACHINE): 1

Truncaded code snippet:
// recommendation_engine.js

const express = require('express');
const bodyParser = require('body-parser');
const fs = require('fs');

const app = express();
app.use(bodyParser.json());

let interacti ...

Loading tokenizer and ModernBERT encoder...
Loading DroidDetect checkpoint/weights...
Missing keys from checkpoint: []
Running classification on example...

Predictions:
  1 - MACHINE_GENERATED: 0.9995
  0 - HUMAN_GENERATED: 0.0005
GT label: 1


### LIME Explanation and Visualization

In [5]:
print('''
In this section we use the lime package's LimeTextExplainer and Explanation
together with a custom heatmap visualization of the result.
''')


In this section we use the lime package's LimeTextExplainer and Explanation
together with a custom heatmap visualization of the result.



In [6]:
explainer = LimeTextExplainer(class_names=labels)
# (documentation: https://lime-ml.readthedocs.io/en/latest/lime.html#lime.lime_text.LimeTextExplainer )

# green-red overlay
def _blend_green_red(score: float, alpha: float = 1.0):
    # we clamp to [0,1] and exponentiate scores
    s = float(np.clip(score, 0.0, 1.0)) ** 1.4  # 0 -> 0, 1 -> 1, mid values squashed
    # ends of our color range
    g = (200, 247, 197)  # light green
    r = (255, 90, 90)  # red
    # linear interpolation
    R = int((1 - s) * g[0] + s * r[0])
    G = int((1 - s) * g[1] + s * r[1])
    B = int((1 - s) * g[2] + s * r[2])
    
    # return rgba string for CSS (alpha = transparency)
    return f"rgba({R},{G},{B},{alpha})"

def lime_heat_from_package(
    input_text: str,
    num_samples: int = 200,
    num_features: int = 1000,  # set high enough to use all tokens
    label: int | None = None,
    batch_size: int = BATCH_SIZE_DEFAULT,
    show_progress: bool = True):
    """
    Runs LIME for one input_text and visualizes evidence for the predicted class in a text heatmap.
    """
    print(f"Running LIME explanation (num_samples={num_samples}, num_features={num_features})...")

    def classification_function(texts):
        # will be  passed to the LIME explainer with all perturbations
        return detect_ai_batched(
            texts,
            batch_size=batch_size,
            show_progress=show_progress,
            desc="LIME: evaluating neighborhood")

    explanation = explainer.explain_instance(
        text_instance=input_text,
        classifier_fn=classification_function,
        labels=list(range(len(labels))),
        num_features=num_features,
        num_samples=num_samples)
    # (documentation: https://lime-ml.readthedocs.io/en/latest/lime.html#lime.explanation.Explanation )

    # use model argmax as default label
    if label is None:
        probs = detect_ai([input_text])[0]
        label = int(probs.argmax())

    indexed = explanation.domain_mapper.indexed_string
    K = indexed.num_words()

    # token-level scores (positive contributions only)
    token_scores = np.zeros(K, dtype=float)
    for token_id, weight in explanation.local_exp[label]:
        token_scores[token_id] = max(weight, 0.0)  # visualize only positive contributions

    # normalize scores for highlighting
    if token_scores.max() > 0:
        token_scores /= token_scores.max()

    # raw input string
    raw = indexed.raw_string()

    # map token scores to char scores to visualize heat
    char_scores = np.zeros(len(raw), dtype=float)
    for i in range(K):  # iterate token indices
        w = indexed.word(i)
        if not w:  # empty/whitespace tokens
            continue
            
        start = indexed.string_position(i)[0]
        end = min(start + len(w), len(raw)) # respect string length at the end
        # assign token score to all its characters
        char_scores[start:end] = np.maximum(char_scores[start:end], token_scores[i])

    # now create all html character highlighted by score 
    spans = [
        f'<span style="background-color:{_blend_green_red(s)}; color:#111; '
        f'font-family:monospace; white-space:pre;">{html.escape(ch)}</span>'
        for ch, s in zip(raw, char_scores)
    ]

    # render heat visualization as html
    html_out = (
        "<div style='border:1px solid #ccc; border-radius:8px; padding:8px; "
        # renders white-spaces and line breakes in our spans list
        "background:#fafafa; font-family:monospace; white-space:pre;'>"
        + "".join(spans)  # join all characters
        + "</div>")
    
    print("Green: No evidence for predicted class.")
    print("Red:   High evidence for the predicted class.")
    display(HTML(html_out))

    print("LIME explanation completed.")
    return explanation, token_scores

# Run LIME and visualize heatmap
exp_pkg, scores_pkg = lime_heat_from_package(
    x_example,
    num_samples=200,
    num_features=100,
    batch_size=BATCH_SIZE_DEFAULT)

Running LIME explanation (num_samples=200, num_features=100)...


Green: No evidence for predicted class.
Red:   High evidence for the predicted class.


LIME explanation completed.


### Custom LIME Version

In this section we apply a custom version of LIME.
To improve the alignment of text segmentation into features with human perception of the code,
we apply a language-specific lexer.
Text segmentation can be controlled via "SEGMENT_MODE" below:
 - "all" uses all tokens as features.
 - "lines" uses line-based feature segmentation.
 - "nl" uses natural language only as lime features, and code tokens are permanently masked.
 - "code" uses code tokens only as lime features, natural language tokens (including comments and strings) are permanently masked.

In [7]:
### Text Segmentation / Lime features

# config
SEGMENT_MODE = "all"  # {"lines", "all", "nl", "code"}
SEGMENT_MAX_LINES_DEMO = 20

_LANG_ALIASES = {
    "C++": "cpp",
    "C#": "csharp",
    "JavaScript": "javascript",
    "TypeScript": "typescript",
    "Python": "python",
    "Java": "java",
    "Go": "go",
    "Rust": "rust",
}

# CSS styles for code visualizations
MASK_COLOR = "rgba(255, 196, 0, 0.55)"  # amber for kept segments for perturbations later
base_css = """
    <style>
    .code-window-pert {  /* box to display code sample with perturbation */
      background: #f5f5f7;
      color: #111111;
      border-radius: 10px;
      padding: 10px 14px;
      font-family: "Fira Code", "Source Code Pro", Menlo, Monaco, monospace;
      font-size: 13px;
      line-height: 1.4;
      border: 1px solid #d0d0d5;
      white-space: pre-wrap;
    }
    .code-window-pert-lines {  /* box to display code sample with perturbation with lines as features */
      background: #282c34;
      color: #abb2bf;
      border-radius: 10px;
      padding: 10px 14px;
      font-family: "Fira Code", "Source Code Pro", Menlo, Monaco, monospace;
      font-size: 13px;
      line-height: 1.4;
      border: 1px solid #181a1f;
    }
    .code-window-pert-lines .lineno {  /* stylish line numbers in "lines" mode */
      color: #888888;
      display: inline-block;
      width: 3.5em;
    }
    .code-window-pert-lines .code-line {  /* here we ensure correct formatting */
      white-space: pre;
    }
    .seg-on {
      background-color: """ + MASK_COLOR + """;
      color: #111111;  /* dark text on light bg used for UNMASKED tokens */
    }
    .seg-off {
      opacity: 0.2;    /* faded text for MASKED tokens */
    }
    .code-hidden {
      background: #000000;
      color: #000000;  /* hide text for permanently masked tokens*/
    }
    </style>
    """

def get_code_lexer(language: str):
    lang = (language or "python").strip()
    lang = _LANG_ALIASES.get(lang, lang.lower())
    try:
        return get_lexer_by_name(lang)
    except Exception:
        return get_lexer_by_name("python")

def _random_pastel_rgba(alpha=0.35):
    r = random.randint(150, 230)
    g = random.randint(150, 230)
    b = random.randint(150, 230)
    return f"rgba({r},{g},{b},{alpha})"

def show_segmentation(code_text: str,
                      language: str = "python",
                      mode: str = "lines",
                      max_lines: int | None = None):
    """
    Return list of segments and optionally display code with segmentation.
    """
    assert mode in {"lines", "all", "nl", "code"}

    ### Lines as Features
    if mode == "lines":
        lexer = get_code_lexer(language)
        formatter = HtmlFormatter(style="default", nowrap=True)
        style_defs = formatter.get_style_defs(".code-window-pert-lines")

        line_segments = code_text.splitlines()
        print(f"[segmentation] mode=lines, K={len(line_segments)} segments")

        if max_lines == 0:
            return line_segments

        display_lines = line_segments
        if max_lines is not None and max_lines > 0:
            display_lines = display_lines[:max_lines]

        line_html_blocks = []
        for i, line in enumerate(display_lines):
            frag = highlight(line, lexer, formatter).rstrip("\n")
            line_html_blocks.append(
                f'<span class="lineno">[L{i}]</span> '
                f'<span class="code-line">{frag}</span>'
            )

        full_html = (
            "<style>" + style_defs + "</style>" +
            '<div class="code-window-pert-lines">' +
            "<br>".join(line_html_blocks) +
            "</div>"
        )
        display(HTML(full_html))
        return line_segments

    ### token-based Features
    lexer = get_code_lexer(language)
    tokens = list(lexer.get_tokens(code_text))

    html_parts: list[str] = []
    segments: list[str] = []
    seg_idx = 0

    for tt, tok in tokens:
        if tt in Token.Text:
            html_parts.append(html.escape(tok))
            continue

        is_nl = (tt in Comment) or (tt in Literal.String)
        visible = False
        bg = None

        ### all tokens are features
        if mode == "all":
            if tok.strip():
                visible = True
                bg = _random_pastel_rgba()

        ### only natural language tokens as unmasked features
        elif mode == "nl":
            if is_nl and tok.strip():
                visible = True
                bg = _random_pastel_rgba()
            else:
                esc = html.escape(tok)
                html_parts.append(f'<span class="code-hidden">{esc}</span>')
                continue

        ### only code tokens as unmasked features
        elif mode == "code":
            if (not is_nl) and tok.strip():
                visible = True
                bg = _random_pastel_rgba()
            else:
                esc = html.escape(tok)
                html_parts.append(f'<span class="code-hidden">{esc}</span>')
                continue

        if visible:
            esc = html.escape(tok)
            html_parts.append(
                f'<span class="seg seg-{seg_idx}" '
                f'style="background-color:{bg}; color:#111111;">{esc}</span>'
            )
            segments.append(tok)
            seg_idx += 1
        else:
            html_parts.append(html.escape(tok))

    print(f"[segmentation] mode={mode}, K={len(segments)} segments")
    print("  examples:", segments[:10])

    if max_lines == 0:
        return segments

    html_body = "".join(html_parts)
    if max_lines is not None and max_lines > 0:
        lines = html_body.split("\n")
        html_body = "\n".join(lines[:max_lines])

    full_html = (
        base_css + '<div class="code-window-pert"><pre>' +
        html_body +
        "</pre></div>"
    )
    display(HTML(full_html))
    return segments


print(f"\n=== Segmentation demo (mode={SEGMENT_MODE}) ===")
segments_demo = show_segmentation(
    x_example,
    language=LANG,
    mode=SEGMENT_MODE,
    max_lines=SEGMENT_MAX_LINES_DEMO,
)


=== Segmentation demo (mode=lines) ===
[segmentation] mode=lines, K=98 segments


In [8]:
### Random Perturbations

def sample_perturbations(num_segments: int,
                         num_perturb: int = 10,
                         p_keep: float = 0.5) -> np.ndarray:
    return np.random.binomial(1, p_keep, size=(num_perturb, num_segments))

def visualize_text_perturbation(code_text: str,
                                perturbation: np.ndarray,
                                language: str = "python",
                                mode: str = "all",
                                max_lines: int | None = None):
    """
    Visualize one perturbed instance for a given segmentation mode.
    """
    assert mode in {"lines", "all", "nl", "code"}
    perturbation = np.asarray(perturbation).astype(int).ravel()
    K = len(perturbation)

    ### Lines as Features
    if mode == "lines":
        lexer = get_code_lexer(language)
        formatter = HtmlFormatter(style="default", nowrap=True)
        style_defs = formatter.get_style_defs(".code-window-pert-lines")

        lines = code_text.splitlines(True)
        assert len(lines) == K, "len(perturbation) must equal #lines"
        print(f"[perturbation] mode=lines, K={len(lines)}")

        display_lines = lines
        if max_lines is not None and max_lines > 0:
            display_lines = lines[:max_lines]

        line_html_blocks = []
        for i, line in enumerate(display_lines):
            frag = highlight(line, lexer, formatter).rstrip("\n")
            z = perturbation[i]
            cls = "seg-on" if z == 1 else "seg-off"
            line_html_blocks.append(
                f'<span class="lineno">[L{i}]</span> '
                f'<span class="code-line {cls}">{frag}</span>'
            )

        full_html = (
            "<style>" + style_defs + "</style>" +
            base_css +
            '<div class="code-window-pert-lines">' +
            "<br>".join(line_html_blocks) +
            "</div>"
        )
        display(HTML(full_html))
        return

    ### token-based Features
    lexer = get_code_lexer(language)
    tokens = list(lexer.get_tokens(code_text))

    html_parts: list[str] = []
    seg_idx = 0

    for tt, tok in tokens:
        if tt in Token.Text:
            html_parts.append(html.escape(tok))
            continue

        esc = html.escape(tok)
        is_nl = (tt in Comment) or (tt in Literal.String)
        
        ### all tokens are features
        if mode == "all":
            if tok.strip():
                z = perturbation[seg_idx]
                seg_idx += 1
                cls = "seg-on" if z == 1 else "seg-off"
                html_parts.append(f'<span class="{cls}">{esc}</span>')
            else:
                html_parts.append(esc)
                
        ### only natural language tokens as unmasked features
        elif mode == "nl":
            if is_nl and tok.strip():
                z = perturbation[seg_idx]
                seg_idx += 1
                cls = "seg-on" if z == 1 else "seg-off"
                html_parts.append(f'<span class="{cls}">{esc}</span>')
            else:
                html_parts.append(f'<span class="code-hidden">{esc}</span>')

        ### only code tokens as unmasked features
        elif mode == "code":
            if (not is_nl) and tok.strip():
                z = perturbation[seg_idx]
                seg_idx += 1
                cls = "seg-on" if z == 1 else "seg-off"
                html_parts.append(f'<span class="{cls}">{esc}</span>')
            else:
                html_parts.append(f'<span class="code-hidden">{esc}</span>')

    print(f"[perturbation] mode={mode}, K={seg_idx} segments")

    html_body = "".join(html_parts)
    if max_lines is not None and max_lines > 0:
        lines = html_body.split("\n")
        html_body = "\n".join(lines[:max_lines])

    full_html = (
        base_css +
        '<div class="code-window-pert"><pre>' +
        html_body +
        "</pre></div>"
    )
    display(HTML(full_html))


print(f"\n=== Perturbation demo (mode={SEGMENT_MODE}) ===")
segments_for_mode = show_segmentation(
    x_example,
    language=LANG,
    mode=SEGMENT_MODE,
    max_lines=0,   # segmentation only here
)

pert_mask_demo = sample_perturbations(
    num_segments=len(segments_for_mode),
    num_perturb=1,
    p_keep=0.5,
)[0]

print("\nHighlighted: Kept segments.")
print("Faded:        Masked segments.\n")

visualize_text_perturbation(
    x_example,
    perturbation=pert_mask_demo,
    language=LANG,
    mode=SEGMENT_MODE,
    max_lines=25,
)


=== Perturbation demo (mode=lines) ===
[segmentation] mode=lines, K=98 segments

Highlighted: Kept segments.
Faded:        Masked segments.

[perturbation] mode=lines, K=98


In [9]:
### LIME's linear classifiers with top-k/heat visualization

# 2 types of vizualizations are very conventional.
# Either the user is presented with the top-k features that drive the classification of the explained class.
# Or we present the raw attribution scores for each segment in the input text.

# config
LIME_MODE       = SEGMENT_MODE
NUM_PERTURB     = 200
KERNEL_WIDTH    = 0.25
LIME_BATCH_SIZE = 32
CODE_TEXT       = x_example
TOP_K_LIST      = [5, 15]

def make_perturbed_text(code_text: str,
                        perturbation: np.ndarray,
                        language: str = "python",
                        mode: str = "all") -> str:
    """
    Build perturbed code snippet from a mask over segments.
    """
    z = np.asarray(perturbation).astype(int).ravel()

    if mode == "lines":
        # Use lexer-based line segments
        #segments = compute_segments(code_text, language, mode="lines")
        segments = code_text.splitlines(True)
        assert len(segments) == len(z)
        return "".join(segment if keep else "\n" for segment, keep in zip(segments, z))

    lexer = get_code_lexer(language)
    tokens = list(lexer.get_tokens(code_text))
    out_parts = []
    seg_idx = 0

    for tt, tok in tokens:
        if tt in Token.Text:
            out_parts.append(tok)
            continue

        is_nl = (tt in Comment) or (tt in Literal.String)

        if mode == "all":
            if tok.strip():
                keep = z[seg_idx]
                seg_idx += 1
                out_parts.append(tok if keep == 1 else "")
            else:
                out_parts.append(tok)

        elif mode == "nl":
            if is_nl and tok.strip():
                keep = z[seg_idx]
                seg_idx += 1
                out_parts.append(tok if keep == 1 else "")
            else:
                out_parts.append(tok)

        elif mode == "code":
            if (not is_nl) and tok.strip():
                keep = z[seg_idx]
                seg_idx += 1
                out_parts.append(tok if keep == 1 else "")
            else:
                out_parts.append(tok)

    assert seg_idx == len(z), f"Used {seg_idx} segments, mask length {len(z)}"
    return "".join(out_parts)

def visualize_text_heat(code_text: str,
                        scores_norm: np.ndarray,
                        language: str = "python",
                        mode: str = "all",
                        max_lines: int | None = None):
    """
    Heat visualization for LIME segments.
    """
    
    # lines mode
    if mode == "lines":
        lexer = get_code_lexer(language)
        formatter = HtmlFormatter(style="default", nowrap=True)
        style_defs = formatter.get_style_defs(".code-window-pert-lines")

        lines = code_text.splitlines(True)
        assert len(lines) == len(scores_norm)
        print(f"[heat] mode=lines, K={len(lines)}")

        display_lines = lines
        if max_lines is not None and max_lines > 0:
            display_lines = lines[:max_lines]

        line_html_blocks = []
        for i, line in enumerate(display_lines):
            frag = highlight(line.rstrip("\n"), lexer, formatter).rstrip("\n")
            s = float(scores_norm[i])
            bg = _blend_green_red(s, alpha=0.9)
            line_html_blocks.append(
                f'<span class="lineno">[L{i}]</span> '
                f'<span class="code-line" style="background-color:{bg};">{frag}</span>'
            )

        full_html = (
            "<style>" + style_defs + "</style>" +
            '<div class="code-window-pert-lines">' +
            "<br>".join(line_html_blocks) +
            "</div>"
        )
        display(HTML(full_html))
        return

    # token-based
    lexer = get_code_lexer(language)
    tokens = list(lexer.get_tokens(code_text))

    html_parts: list[str] = []
    seg_idx = 0

    for tt, tok in tokens:
        if tt in Token.Text:
            html_parts.append(html.escape(tok))
            continue

        esc = html.escape(tok)
        is_nl = (tt in Comment) or (tt in Literal.String)

        if mode == "all":
            if tok.strip():
                s = float(scores_norm[seg_idx]) if seg_idx < len(scores_norm) else 0.0
                seg_idx += 1
                bg = _blend_green_red(s, alpha=0.9)
                html_parts.append(
                    f'<span style="background-color:{bg}; color:#111111;">{esc}</span>'
                )
            else:
                html_parts.append(esc)

        elif mode == "nl":
            if is_nl and tok.strip():
                s = float(scores_norm[seg_idx]) if seg_idx < len(scores_norm) else 0.0
                seg_idx += 1
                bg = _blend_green_red(s, alpha=0.9)
                html_parts.append(
                    f'<span style="background-color:{bg}; color:#111111;">{esc}</span>'
                )
            else:
                html_parts.append('<span class="code-hidden">' + esc + "</span>")

        elif mode == "code":
            if (not is_nl) and tok.strip():
                s = float(scores_norm[seg_idx]) if seg_idx < len(scores_norm) else 0.0
                seg_idx += 1
                bg = _blend_green_red(s, alpha=0.9)
                html_parts.append(
                    f'<span style="background-color:{bg}; color:#111111;">{esc}</span>'
                )
            else:
                html_parts.append('<span class="code-hidden">' + esc + "</span>")

    print(f"[heat] used {seg_idx} scores out of {len(scores_norm)}")

    html_body = "".join(html_parts)
    if max_lines is not None and max_lines > 0:
        lines = html_body.split("\n")
        html_body = "\n".join(lines[:max_lines])

    full_html = (
        base_css + '<div class="code-window-pert"><pre>' +
        html_body +
        "</pre></div>"
    )
    display(HTML(full_html))


print(f"\n=== LIME demo (mode={LIME_MODE}) ===")
print("[LIME] 1) Segmenting text...")
segments = show_segmentation(
    CODE_TEXT,
    language=LANG,
    mode=LIME_MODE,
    max_lines=0,
)
K = len(segments)
print(f"[LIME] mode={LIME_MODE}, K={K} segments")

print("[LIME] 2) Sampling perturbations...")
perturbations = sample_perturbations(
    num_segments=K,
    num_perturb=NUM_PERTURB,
    p_keep=0.5,
)
print(f"[LIME] perturbation matrix shape: {perturbations.shape}")

print("[LIME] 3) Building perturbed texts...")
pert_texts = [
    make_perturbed_text(CODE_TEXT, pert, language=LANG, mode=LIME_MODE)
    for pert in tqdm(perturbations, desc="[LIME] building perturbed texts")
]

print("[LIME] 4) Running classifier on perturbations...")
pred_batches = []
for i in tqdm(range(0, len(pert_texts), LIME_BATCH_SIZE), desc="[LIME] running classifier"):
    batch = pert_texts[i:i + LIME_BATCH_SIZE]
    pred_batch = detect_ai_batched(
        batch,
        batch_size=BATCH_SIZE_DEFAULT,
        show_progress=False,
    )
    pred_batches.append(pred_batch)

predictions = np.vstack(pred_batches)
print("[LIME] predictions shape:", predictions.shape)

print("[LIME] 5) Original prediction and class to explain...")
orig_probs = detect_ai([CODE_TEXT])[0]
class_to_explain = int(orig_probs.argmax())
print("  class_to_explain:", class_to_explain, "prob:", float(orig_probs[class_to_explain]))

print("[LIME] 6) Distances, kernel weights, local linear model...")
original_z = np.ones(K)[np.newaxis, :]
distances = pairwise_distances(
    perturbations,
    original_z,
    metric="cosine",
).ravel()
weights = np.sqrt(np.exp(-(distances**2) / KERNEL_WIDTH**2))

lr = LinearRegression()
lr.fit(
    X=perturbations,
    y=predictions[:, class_to_explain],
    sample_weight=weights,
)
coeff = lr.coef_      # (K,)
scores = np.maximum(coeff, 0.0)
scores_norm = scores / scores.max() if scores.max() > 0 else scores

print("[LIME] local surrogate fitted.")
print("  coeff (first 10):", coeff[:10])

print("[LIME] 7) Top-k visualizations via perturbation...")
sorted_idx = np.argsort(scores)  # ascending

for k in TOP_K_LIST:
    k = min(k, K)
    top_ids = sorted_idx[-k:]
    mask = np.zeros(K, dtype=int)
    mask[top_ids] = 1
    print(f"\n=== Top {k} segments (mode={LIME_MODE}, class={class_to_explain}) ===")
    visualize_text_perturbation(
        CODE_TEXT,
        perturbation=mask,
        language=LANG,
        mode=LIME_MODE,
        max_lines=2500,
    )

print("\n[LIME] 8) Full heatmap view...")
visualize_text_heat(
    CODE_TEXT,
    scores_norm,
    language=LANG,
    mode=LIME_MODE,
    max_lines=3000,
)


=== LIME demo (mode=lines) ===
[LIME] 1) Segmenting text...
[segmentation] mode=lines, K=98 segments
[LIME] mode=lines, K=98 segments
[LIME] 2) Sampling perturbations...
[LIME] perturbation matrix shape: (200, 98)
[LIME] 3) Building perturbed texts...


[LIME] building perturbed texts: 100%|███████████████████████████████████████████████████████████████| 200/200 [00:00<00:00, 34698.08it/s]


[LIME] 4) Running classifier on perturbations...


[LIME] running classifier: 100%|████████████████████████████████████████████████████████████████████████████| 7/7 [00:20<00:00,  2.95s/it]


[LIME] predictions shape: (200, 2)
[LIME] 5) Original prediction and class to explain...
  class_to_explain: 1 prob: 0.9994869232177734
[LIME] 6) Distances, kernel weights, local linear model...
[LIME] local surrogate fitted.
  coeff (first 10): [-0.47488137  0.07304095  0.08167011  0.04129863  0.01009179  0.01768369
 -0.00115921  0.02092557 -0.08437009 -0.0012389 ]
[LIME] 7) Top-k visualizations via perturbation...

=== Top 5 segments (mode=lines, class=1) ===
[perturbation] mode=lines, K=98



=== Top 15 segments (mode=lines, class=1) ===
[perturbation] mode=lines, K=98



[LIME] 8) Full heatmap view...
[heat] mode=lines, K=98


## Live LIME visualization

**Live evidence updates**

- Each step we sample a random binary mask over our text segments, drop masked segments and re-run the text-classifier.
- Every UPDATE_EVERY steps re-fit a lightweight kernel-weighted local linear model ($\phi$) to all the accumulated samples.
- Heat per segment $s_i^{(t)}$ represents the normalized non-negative coefficient (absolute coefficient for lasso).

$$
\tilde s_i^{(t)}=\frac{\phi(\beta_i^{(t)})}{\max_j \phi(\beta_j^{(t)})},
\quad
\phi(b)=
\begin{cases}
\max(b,0) & \text{linear/ridge}\\
|b| & \text{lasso}
\end{cases}
$$

- Colors jump early (few samples), then stabilize as \(t\) grows.

Give it a try!<br>
You see a segment's actual score and the difference from the previous update when you hover over the tokens.

### LIME x DroidCollection UI

In [10]:
### Live Config defaults
LIVE_MODE          = "all"
NUM_PERTURB_LIVE   = 2000
KERNEL_WIDTH_LIVE  = 0.25
UPDATE_EVERY       = 30
REG_TYPE           = "linear"  # "linear", "ridge", "lasso"
CLASS_MODE         = "pred"  # "pred" (predicted) or "flip" (other class)
INFO_COLUMN        = "Generator"

### Widgets for heat overlay, notificatioins and run button
run_button = widgets.Button(
    description="Run LIME (live)",
    button_style="primary",
    tooltip="Run LIME on this snippet and show live heat")

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=NUM_PERTURB_LIVE,
    description="Samples",
    bar_style="info")

# 3 boxes for infos, logs and the actual viz
sample_info_out = widgets.Output(
    layout=dict(
        border="1px solid #ddd",
        height="180px",
        overflow_y="auto",
        padding="4px"))

log_out = widgets.Output(
    layout=dict(
        border="1px solid #ccc",
        height="140px",
        overflow_y="auto",
        padding="4px"))

viz_out = widgets.Output(
    layout=dict(
        border="1px solid #ccc",
        padding="4px"))

# CSS rendering of text visualization
LIVE_HEAT_CSS = """
<style>
.lime-live-code {
  background: #f5f5f7;
  color: #111111;
  padding: 10px 14px;
  font-family: "Fira Code", "Source Code Pro", Menlo, Monaco, monospace;
  font-size: 13px;
  line-height: 1.4;
  white-space: pre-wrap;
}
.lime-live-seg {
  cursor: default;
}
.lime-live-seg-up {
  outline: 1px solid #ff3b30;  /* a red box signals increased weight at current iteration */
}
.lime-live-seg:hover {
  outline: 1px solid #999999;  /* a thin gray box displays the mean score on hover */
}
</style>
"""

def visualize_text_heat_live(
    code_text: str,
    scores_norm: np.ndarray,
    prev_scores_norm: np.ndarray | None,
    language: str = "python",
    mode: str = "all"):  # "lines", "all", "nl" or "code"
    """
    Heat visualization for LIME segments.
    Features and visualization depend on the mode:
    - lines: one segment per line
    - others: token segments
    """

    # The ui displays each segments score, iterative delta 
    # and a flag for segments whole score increased
    def _seg_stats(idx: int):
        score = float(scores_norm[idx]) if idx < len(scores_norm) else 0.0
        if prev_scores_norm is not None:
            prev = float(prev_scores_norm[idx])
        else:
            prev = 0.0
        delta = score - prev
        increased = delta > 1e-6
        return score, delta, increased

    ### Lines as Features
    if mode == "lines":
        lexer = get_code_lexer(language)
        formatter = HtmlFormatter(style="default", nowrap=True)
        style_defs = formatter.get_style_defs(".code-window-heat-lines")

        lines = code_text.splitlines(True)
        line_html_blocks = []
        for i, line in enumerate(lines):
            score, delta, increased = _seg_stats(i)
            bg = _blend_green_red(score, alpha=0.9)

            frag = highlight(line.rstrip("\n"), lexer, formatter).rstrip("\n")
            cls = "code-line"
            if increased:
                cls += " lime-live-seg-up"

            line_html_blocks.append(
                f'<span class="lineno">[L{i}]</span> '
                f'<span class="{cls}" title="ŝ={score:.3f}, s\'={delta:+.3f}" '
                f'style="background-color:{bg};">{frag}</span>')

        full_html = f"<style>{style_defs}</style>{LIVE_HEAT_CSS}" \
                    '<div class="code-window-heat-lines">' + \
                    "<br>".join(line_html_blocks) + \
                    "</div>"
        display(HTML(full_html))
        return

    ### token-based Features
    lexer = get_lexer_by_name(language)
    tokens = list(lexer.get_tokens(code_text))

    parts = []
    seg_idx = 0

    for tt, tok in tokens:
        if tt in Token.Text or not tok.strip():
            parts.append(html.escape(tok))  # preserve layout
            continue

        esc = html.escape(tok)
        is_nl = (tt in Comment) or (tt in Literal.String)

        is_segment = (
            (mode == "all") or
            (mode == "nl" and is_nl) or
            (mode == "code" and not is_nl)
        )

        if is_segment:
            score, delta, increased = _seg_stats(seg_idx)
            bg = _blend_green_red(score, alpha=0.9)
            cls = "lime-live-seg"
            if increased:
                cls += " lime-live-seg-up"

            parts.append(
                f'<span class="{cls}" title="ŝ={score:.3f}, s\'={delta:+.3f}" '
                f'style="background-color:{bg}; color:#111111;">{esc}</span>')
            seg_idx += 1
        else:
            # Non-segments are masked as classifier input and not rendered
            parts.append("")

    full_html = LIVE_HEAT_CSS + \
                '<div class="lime-live-code"><pre>' + \
                "".join(parts) + \
                "</pre></div>"
    display(HTML(full_html))


    
### Text/feature Segmention
def compute_segments(code_text: str, language: str, mode: str) -> list[str]:
    assert mode in {"lines", "all", "nl", "code"}
    lexer = get_lexer_by_name((language or "python").lower())
    segments = []

    if mode == "lines":
        current_line = ""
        for tt, tok in lexer.get_tokens(code_text):
            current_line += tok
            if "\n" in tok:
                segments.append(current_line)
                current_line = ""
        if current_line.strip():  # trailing line without newline
            segments.append(current_line)
        return segments

    for tt, tok in lexer.get_tokens(code_text):
        if tt in Token.Text:
            continue
        if not tok.strip():
            continue
        is_nl = (tt in Comment) or (tt in Literal.String)
        if mode == "all":
            ### all tokens are features
            segments.append(tok)
        elif mode == "nl" and is_nl:
            ### only natural language tokens as unmasked features
            segments.append(tok)
        elif mode == "code" and (not is_nl):
            ### only code tokens as unmasked features
            segments.append(tok)
    return segments


### Dataset navigation controls
num_samples = len(ds)

sample_idx_slider = widgets.IntSlider(
    value=sample_idx,
    min=0,
    max=num_samples - 1,
    step=1,
    description="Sample",
    continuous_update=False)

prev_btn = widgets.Button(
    description="◀",
    tooltip="Previous sample",
    layout=widgets.Layout(width="40px"))

next_btn = widgets.Button(
    description="▶",
    tooltip="Next sample",
    layout=widgets.Layout(width="40px"))

column_dropdown = widgets.Dropdown(
    options=list(ds.column_names),
    value=INFO_COLUMN,
    description="Info Column")

### LIME configuration controls
mode_dropdown = widgets.Dropdown(
    options=[
        ("All tokens", "all"),
        ("Lines", "lines"),
        ("NL only", "nl"),
        ("Code only", "code")],
    value=LIVE_MODE,
    description="Mode")

num_perturb_widget = widgets.IntText(
    value=NUM_PERTURB_LIVE,
    description="#samples",
    min=10)

kernel_width_widget = widgets.FloatText(
    value=KERNEL_WIDTH_LIVE,
    description="kernel σ")

update_every_widget = widgets.IntText(
    value=UPDATE_EVERY,
    description="update every",
    min=1)

params_info_out = widgets.Output(
    layout=dict(
        border="1px solid #ddd",
        height="80px",
        overflow_y="auto",
        padding="4px"))

### Regression controls
reg_type_dropdown = widgets.ToggleButtons(
    options=[
        ("Linear", "linear"),
        ("Ridge (L2)", "ridge"),
        ("Lasso (L1)", "lasso")],
    value=REG_TYPE,
    layout=widgets.Layout(width="100%"))

# Custom code input 
custom_code_area = widgets.Textarea(
    value="",
    placeholder="Paste your code snippet here...",
    description="Custom",
    layout=widgets.Layout(width="100%", height="140px"))

use_custom_btn = widgets.Button(
    description="Use custom",
    tooltip="Use this code instead of dataset sample",
    button_style="")

# Control over class to explain
class_mode_toggle = widgets.ToggleButtons(
    options=[
        ("Predicted class", "pred"),
        ("Flip class", "flip")],
    value="pred",
    description="Explain",
    layout=widgets.Layout(width="100%"))

def on_class_mode_change(change):
    if change["name"] == "value":
        global CLASS_MODE
        CLASS_MODE = change["new"]

class_mode_toggle.observe(on_class_mode_change, names="value")


### UI State and utils
def apply_live_params():
    """Push widget values into the global config used by run_live_lime."""
    global LIVE_MODE, NUM_PERTURB_LIVE, KERNEL_WIDTH_LIVE, UPDATE_EVERY

    LIVE_MODE           = mode_dropdown.value
    NUM_PERTURB_LIVE    = int(num_perturb_widget.value)
    KERNEL_WIDTH_LIVE   = float(kernel_width_widget.value)
    UPDATE_EVERY        = max(1, int(update_every_widget.value))

    # keep progress bar consistent with #samples
    progress_bar.max = NUM_PERTURB_LIVE
    progress_bar.value = 0

    params_info_out.clear_output()
    with params_info_out:
        print("[live config]")
        print(f"  mode         = {LIVE_MODE}")
        print(f"  num_perturb  = {NUM_PERTURB_LIVE}")
        print(f"  kernel_width = {KERNEL_WIDTH_LIVE}")
        print(f"  update_every = {UPDATE_EVERY}")

def apply_regression_type(new_value: str):
    global REG_TYPE
    REG_TYPE = new_value

def refresh_sample_view(idx: int | None = None):
    """
    Update:
      - x_example, y_example, LANG (globals used by live LIME)
      - sample_info_out (label, prediction, column content)
      - viz_out (initial all-green view, no LIME yet)
    """
    global x_example, y_example, LANG, sample_idx

    if idx is None:
        idx = int(sample_idx_slider.value)
    else:
        idx = int(idx)
        sample_idx_slider.value = idx  # slider update

    row = ds[idx]

    # update globals for downstream cells
    x_example = row[CODE_COL]
    y_example = row["y"]
    LANG = row.get("Language", "python")
    sample_idx = idx

    # reset class mode for new sample (default = predicted class)
    class_mode_toggle.value = "pred"   # updates CLASS_MODE via callback

    # model prediction on this snippet
    probs = detect_ai([x_example])[0]
    pred_idx = int(probs.argmax())
    pred_label = labels[pred_idx]
    pred_prob = float(probs[pred_idx])

    # segments for current live mode (for initial all-green view)
    segs = compute_segments(x_example, language=LANG, mode=LIVE_MODE)
    K = len(segs)
    zero_scores = np.zeros(K, dtype=float)

    # update sample info box (LEFT)
    sample_info_out.clear_output()
    with sample_info_out:
        print(f"Sample index: {idx}")
        print(f"Dataset label y: {y_example} ({labels[y_example] if y_example in (0,1) else 'other'})")
        print(f"Model prediction: {pred_label} (p={pred_prob:.3f})")
        print(f"Language: {LANG}")
        print("")
        sel_col = column_dropdown.value
        print(f"{sel_col}:")
        print(row[sel_col])

    # initial visualization in the SAME viz_out used by live LIME
    viz_out.clear_output(wait=True)
    with viz_out:
        print("[LIVE] initial view (no LIME yet)")
        visualize_text_heat_live(
            x_example,
            scores_norm=zero_scores,
            prev_scores_norm=None,
            language=LANG,
            mode=LIVE_MODE,
        )

def use_custom_snippet(_):
    global x_example, y_example, LANG, sample_idx

    txt = custom_code_area.value.strip()
    if not txt:
        log_out.clear_output(wait=True)
        with log_out:
            print("[CUSTOM] No code provided. Please paste some code first.")
        return

    # just treat as a non-dataset sample
    x_example = txt
    y_example = -1
    LANG = "python"   # currently fixed to python
    sample_idx = -1

    # reset class mode to predicted
    class_mode_toggle.value = "pred"

    # mark sample as custom in info box
    sample_info_out.clear_output()
    with sample_info_out:
        print("Sample: CUSTOM INPUT")
        print("Dataset label y: n/a")
        print("Model prediction: (will be computed when running LIME)")
        print(f"Language: {LANG}")
        print("")
        print("Code:")
        print(x_example)

    # initial green viz for this snippet
    segs = compute_segments(x_example, language=LANG, mode=LIVE_MODE)
    zero_scores = np.zeros(len(segs), dtype=float)

    viz_out.clear_output(wait=True)
    with viz_out:
        print("[LIVE] initial view (custom snippet, no LIME yet)")
        visualize_text_heat_live(
            x_example,
            scores_norm=zero_scores,
            prev_scores_norm=None,
            language=LANG,
            mode=LIVE_MODE,
        )

    # clear old log and shiw status
    log_out.clear_output(wait=True)
    with log_out:
        print("[CUSTOM] Loaded custom snippet.")
        print("Click 'Run LIME (live)' to explain this code.")

use_custom_btn.on_click(use_custom_snippet)

### Control callbacks 
def on_sample_change(change):
    if change["name"] == "value":
        apply_live_params()
        refresh_sample_view(change["new"])

def on_prev_clicked(_):
    new_idx = max(0, sample_idx_slider.value - 1)
    if new_idx != sample_idx_slider.value:
        sample_idx_slider.value = new_idx  # triggers on_sample_change

def on_next_clicked(_):
    new_idx = min(num_samples - 1, sample_idx_slider.value + 1)
    if new_idx != sample_idx_slider.value:
        sample_idx_slider.value = new_idx  # triggers on_sample_change

def on_column_change(change):
    if change["name"] == "value":
        refresh_sample_view(sample_idx_slider.value)

def on_params_change(change):
    if change["name"] == "value":
        apply_live_params()
        refresh_sample_view(sample_idx_slider.value)

def on_reg_type_change(change):
    if change["name"] == "value":
        apply_regression_type(change["new"])

sample_idx_slider.observe(on_sample_change, names="value")
prev_btn.on_click(on_prev_clicked)
next_btn.on_click(on_next_clicked)
column_dropdown.observe(on_column_change, names="value")

mode_dropdown.observe(on_params_change, names="value")
num_perturb_widget.observe(on_params_change, names="value")
kernel_width_widget.observe(on_params_change, names="value")
update_every_widget.observe(on_params_change, names="value")

reg_type_dropdown.observe(on_reg_type_change, names="value")


# LIVE loop
def run_live_lime(_):
    log_out.clear_output()
    viz_out.clear_output()
    progress_bar.value = 0
    progress_bar.bar_style = "info"

    try:
        # Segmentation, number of segments K
        segments = show_segmentation(
            x_example,
            language=LANG,
            mode=LIVE_MODE,
            max_lines=0)  # segmentation only, no display
        K = len(segments)

        # Perturbations and kernel weights
        perturbations = sample_perturbations(
            num_segments=K,
            num_perturb=NUM_PERTURB_LIVE,
            p_keep=0.5)
        original_z = np.ones((1, K))
        distances = pairwise_distances(
            perturbations,
            original_z,
            metric="cosine").ravel()
        weights_all = np.sqrt(np.exp(-(distances**2) / KERNEL_WIDTH_LIVE**2))

        # Class to explain (based on predicted class and the CLASS_MODE setting)
        orig_probs = detect_ai([x_example])[0]
        pred_idx = int(orig_probs.argmax())
        if CLASS_MODE == "pred":
            class_to_explain = pred_idx
        else:
            # binary flip
            # (Here we assume binary labels {0,1}
            # TODO: you'd need change this if you want to try out a different DroidDetector model)
            class_to_explain = 1 - pred_idx

        gt_name = labels[y_example] if y_example in (0, 1) else "other"

        header_lines = [
            f"[LIVE LIME] mode={LIVE_MODE}",
            f"[LIVE LIME] K={K} segments",
            f"[LIVE LIME] regression: {REG_TYPE}",
            f"  predicted class: {pred_idx} {labels[pred_idx]} (p={orig_probs[pred_idx]:.3f})",
            f"  explained class: {class_to_explain} {labels[class_to_explain]}",
            f"  true class: {y_example} {gt_name}",
            f"[LIVE LIME] running {NUM_PERTURB_LIVE} perturbations...",
            ""]

        def update_log(progress: int):
            log_out.clear_output(wait=True)
            with log_out:
                for line in header_lines:
                    print(line)
                print(f"[LIVE LIME] samples used: {progress}/{NUM_PERTURB_LIVE}")

        update_log(0)

        predictions_all = []
        prev_scores = None  # for delta highlighting

        # live loop
        for i in range(NUM_PERTURB_LIVE):
            z = perturbations[i]

            pert_text = make_perturbed_text(
                x_example,
                z,
                language=LANG,
                mode=LIVE_MODE)

            pred = detect_ai([pert_text])[0]
            predictions_all.append(pred)

            n_used = i + 1
            progress_bar.value = n_used

            if (n_used % UPDATE_EVERY != 0) and (n_used != NUM_PERTURB_LIVE):
                continue

            # fitting local regressor on current stack of samples
            X_used = perturbations[:n_used]
            y_used = np.vstack(predictions_all)[:n_used, class_to_explain]
            w_used = weights_all[:n_used]

            # choose regression type
            if REG_TYPE == "linear":
                lr = LinearRegression()
            elif REG_TYPE == "ridge":
                lr = Ridge(alpha=1.0)
            elif REG_TYPE == "lasso":
                lr = Lasso(alpha=1e-3, max_iter=10000)
            else:
                lr = LinearRegression()

            lr.fit(X_used, y_used, sample_weight=w_used)

            coeff = lr.coef_  # shape (K,)
            #scores = np.maximum(coeff, 0.0)

            if REG_TYPE == "lasso":
                scores = np.abs(coeff)  # use magnitude for L1
            else:
                scores = np.maximum(coeff, 0.0)
            
            scores_norm = scores / scores.max() if scores.max() > 0 else scores

            # update log and heat view
            update_log(n_used)

            viz_out.clear_output(wait=True)
            with viz_out:
                print(f"[LIVE LIME] live heat (samples used: {n_used}/{NUM_PERTURB_LIVE})")
                print("Green: No evidence (coefficient 0)")
                print("Red:   High evidence (coefficient 1)")
                visualize_text_heat_live(
                    x_example,
                    scores_norm,
                    prev_scores_norm=prev_scores,
                    language=LANG,
                    mode=LIVE_MODE)

            prev_scores = scores_norm.copy()

        progress_bar.bar_style = "success"
        update_log(NUM_PERTURB_LIVE)

    except Exception:
        with log_out:
            print("[LIVE LIME] ERROR:")
            import traceback
            traceback.print_exc()
        progress_bar.bar_style = "danger"


run_button.on_click(run_live_lime)


### Layout
# fill initial widget current globals
apply_live_params()
refresh_sample_view(sample_idx_slider.value)
apply_regression_type(reg_type_dropdown.value)

# Control panel RIGHT in a vertical bar
nav_row = widgets.HBox([prev_btn, sample_idx_slider, next_btn])

right_panel = widgets.VBox(
    [
        widgets.HTML("<b>Dataset & sample</b>"),
        nav_row,
        column_dropdown,
        widgets.HTML("<hr><b>Live LIME settings</b>"),
        mode_dropdown,
        num_perturb_widget,
        kernel_width_widget,
        update_every_widget,
        params_info_out,
        widgets.HTML("<hr><b>Regularization</b>"),
        reg_type_dropdown,
        widgets.HTML("<hr><b>Class to explain</b>"),  
        class_mode_toggle,  
        widgets.HTML("<hr><b>Custom code</b>"),
        custom_code_area,                 
        use_custom_btn,
    ],
    layout=widgets.Layout(width="370px"))

# LEFT panel includes sample info, run button, log and viz
left_panel = widgets.VBox(
    [
        sample_info_out,
        widgets.HBox([run_button, progress_bar]),
        log_out,
        viz_out],
    layout=widgets.Layout(flex="1 1 auto"))

ui_frame = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(width="100%"))

display(ui_frame)

### LIME instability

Note that this LIME demonstration is not really "live". <br>
Each perturbation and associated prediction is computed iteratively,<br>
and we also redo the entire regression at each step, while accumulating more data.

LIME is known to produce **unstable** explanations in high-dimensional spaces like text/code. Explanations can shift significantly across runs or with small config changes.

> “\[It was\] demonstrated that the random generation of perturbations results in instability in the generated explanations.”  
> — *Zhou et al., 2021* ([S-LIME](https://arxiv.org/abs/2106.07875))

> “LIME can vary significantly due to minor changes in the input data, perturbation process, sampling, repeated runs, or the underlying model”  
> — *Knab et al., 2025* ([Which LIME Should I Trust?](https://arxiv.org/html/2503.24365v1))

You might observe such drift in our "live" droid-detection setup as well. 

### Custom example 

Below is a small code block generated with GPT-4o that you can use as a curstom input example (lowest section of the ui controlls above):

In [11]:
'''
import random
from typing import List, Dict
import matplotlib.pyplot as plt


def generate_synthetic_dataset_with_fixed_features(
    total_number_of_records_to_generate: int
) -> List[Dict[str, float]]:
    """
    Generates a synthetic dataset with three arbitrary features using random values.
    """
    synthetic_dataset_container: List[Dict[str, float]] = []
    for record_index in range(total_number_of_records_to_generate):
        synthetic_record: Dict[str, float] = {
            "feature_one_alpha": random.uniform(0.0, 1.0),
            "feature_two_beta": random.uniform(50.0, 150.0),
            "feature_three_gamma": random.gauss(0.0, 1.0)
        }
        synthetic_dataset_container.append(synthetic_record)
    return synthetic_dataset_container


def extract_feature_column_from_dataset(
    dataset_in_dictionary_form: List[Dict[str, float]], feature_key_name: str
) -> List[float]:
    """
    Extracts a list of values for a specified feature from the dataset.
    """
    feature_values_container: List[float] = []
    for data_record in dataset_in_dictionary_form:
        feature_values_container.append(data_record.get(feature_key_name, 0.0))
    return feature_values_container


def compute_mean_of_numeric_value_list(
    list_of_numeric_values: List[float]
) -> float:
    """
    Computes the arithmetic mean of a list of numeric values.
    """
    if not list_of_numeric_values:
        return 0.0
    cumulative_sum_total: float = sum(list_of_numeric_values)
    number_of_elements: int = len(list_of_numeric_values)
    return cumulative_sum_total / number_of_elements


def display_histogram_for_feature_values(
    numeric_feature_values: List[float], label_of_feature: str
) -> None:
    """
    Displays a histogram of numeric feature values using matplotlib.
    """
    plt.hist(numeric_feature_values, bins=20, edgecolor='black')
    plt.title(f"Histogram for {label_of_feature}")
    plt.xlabel(label_of_feature)
    plt.ylabel("Frequency")
    plt.show()


def execute_complete_data_pipeline_with_visualizations() -> None:
    """
    Executes the synthetic data generation, feature extraction, mean computation, 
    and histogram visualization steps sequentially.
    """
    number_of_records_to_generate: int = 50
    generated_dataset = generate_synthetic_dataset_with_fixed_features(number_of_records_to_generate)

    list_of_feature_keys: List[str] = [
        "feature_one_alpha",
        "feature_two_beta",
        "feature_three_gamma"
    ]

    for current_feature_key in list_of_feature_keys:
        extracted_feature_values = extract_feature_column_from_dataset(
            generated_dataset, current_feature_key
        )
        mean_of_feature_values = compute_mean_of_numeric_value_list(
            extracted_feature_values
        )
        print(
            f"The arithmetic mean of {current_feature_key} is {mean_of_feature_values:.4f}"
        )
        display_histogram_for_feature_values(extracted_feature_values, current_feature_key)


if __name__ == "__main__":
    execute_complete_data_pipeline_with_visualizations()
''';